# Prepare finetuning data

In [ ]:
import torch
import unirna_tf
import pandas as pd
import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
import torch.nn as nn
from torch.nn import MSELoss

class finetune_dataset(Dataset):
    def __init__(self, dataset, tokenizer):
        self.dataset = dataset
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        tokenized_sequence = self.tokenizer(
            self.dataset[idx]["sequence"], padding="max_length", truncation=True, max_length=1024, return_tensors="pt"
        )

        return {
            "input_ids": tokenized_sequence["input_ids"].squeeze(),
            "attention_mask": tokenized_sequence["attention_mask"].squeeze(),
            "label": self.dataset[idx]["label"],
        }

In [2]:
csv_dataset = pd.read_csv("train.csv").to_dict(orient="records")
model_weight_path = "../weights/unirna_L16"

tokenizer = AutoTokenizer.from_pretrained(model_weight_path)
dataset = finetune_dataset(csv_dataset, tokenizer)
trainset, testset = torch.utils.data.random_split(
    dataset, [int(len(dataset) * 0.8), len(dataset) - int(len(dataset) * 0.8)]
)

# Prepare pretrained models

In [3]:
class CustomLanguageModelHead(nn.Module):
    def __init__(self, model_backbone, hidden_size):
        super(CustomLanguageModelHead, self).__init__()

        self.bert_base = model_backbone
        self.decoder = nn.Linear(hidden_size, 1)

    def forward(self, input_ids, attention_mask=None, **kwargs):
        outputs = self.bert_base(input_ids, attention_mask=attention_mask)
        sequence_output = outputs["pooler_output"]  # we use cls token as the representation of the whole sequence
        prediction_scores = self.decoder(sequence_output)
        return prediction_scores

In [4]:
# change hidden size to the model you use
# for L8 model, use 512, for L12 model, use 768, for L16 model, use 1024

model = AutoModel.from_pretrained(model_weight_path)
model_with_lm_head = CustomLanguageModelHead(model, hidden_size=1024)

# Model training

In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"
optimizer = torch.optim.Adam(model_with_lm_head.parameters(), lr=1e-4)
train_loader = DataLoader(trainset, batch_size=8, shuffle=True)

In [ ]:
model_with_lm_head.train()
model_with_lm_head.to(device)

for epoch in range(3):
    tqdmer = tqdm.tqdm(train_loader, total=len(train_loader))
    mseloss = MSELoss()
    for data in tqdmer:
        label = data["label"].to(device, dtype=torch.float)
        input_ids = data["input_ids"].to(device, dtype=torch.long)
        attention_mask = data["attention_mask"].to(device, dtype=torch.long)
        outputs = model_with_lm_head(input_ids=input_ids, attention_mask=attention_mask)

        loss = mseloss(outputs, label)
        tqdmer.set_postfix(epoch=epoch, loss=loss)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()